In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json
import time
import os
import pickle
import multiprocessing as mp
from importlib import reload


import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/haman/mf-temporary/MeanFieldTester')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')

import codes.controller as rw
from codes.plotting import fig_plots
from codes import plotting as ax_plt
import codes.cell_library as cells
import codes.neuron_simulation as ns
import codes.meanfield_simulation as mfs
import codes.network_simulation as nets
import codes.transfer_function as tf
from codes import utils

from codes.tvb_models.models import Zerlaut_adaptation_first_order


In [ ]:
workflow_params = {
    "single_neuron_simulations" : {
        "simulate_single_neuron" : False,
        "load_single_neuron" : True,
        "fix_nu_out" : True,
        "nu_e_range" : [0, 60, 16],
        "nu_i_range" : [0, 60, 16],
        "nu_out_range" : [0, 60, 16],
        "max_nu_e_step" : 4,
        "simulation_time" : 5000,
        "averaging_window" : 2000,
        "time_step" : 0.1,
        "seed" : 42,
        "n_runs" : 5,
        "cpus" : 16
    },
    "transfer_functions" : {
        "fit_transfer_function" : True,
        "square_terms" : True,
        "log_term" : False,
        "adaptation" : True,
        "fit_with_w" : True,
        "nu_out_min" : 0.0,
        "nu_out_max" : 200.0,
        "V_eff_fitting" : {
            "method" : "SLSQP",
            "options" : {
                "ftol" : 5e-9,
                "disp" : False,
                "maxiter" : 10000
            }
        },
        "TF_fitting" : {
            "method" : "nelder-mead",
            "options" : {
                "fatol": 5e-9,
                "disp" : False,
                "maxiter" : 10000
            }
        }
    },
    "network_simulations" : {
        "load" : True,
        "timestep" : 0.1,
        "seed" : 42,
        "n_runs" : 5,
        "smoothing_function" : "sliding_window",
        "smoothing_constant" : 50,
        "smoothing_kwargs" : {}
    },
    "mf_model" : {
        "T" : 40.0,
        "tvb_model" : "DiVolo_Tsodyks_second_order",
        "mf_init" : {
            "E" : [0.000, 0.000],
            "I" : [0.00, 0.00],
            "C_ee" : [0.0000005, 0.0000005],
            "C_ei" : [0.0000005, 0.0000005],
            "C_ii" : [0.0000005, 0.0000005],
            "W_e" : [50.0, 50.0],
            "W_i" : [0.0, 0.0],
            "X_e" : [1.0, 1.0],
            "Y_e" : [0.0, 0.0],
            "X_i" : [1.0, 1.0],
            "Y_i" : [0.0, 0.0],
            "noise" : [0.0, 0.0],
            "stimulus" : [0.0, 0.0]
        }
    }
}


In [ ]:
divolo_static_spont_drives = np.linspace(1.5, 7, 12)

STIMULI = {
    "SpontActivity1.5" : {
        "pattern" : "NoStimulus",
        "stim_pars" : {},
        "drive_rate" : 1.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "SpontActivity2.5" : {
        "pattern" : "NoStimulus",
        "stim_pars" : {},
        "drive_rate" : 2.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "SpontActivity3.5" : {
        "pattern" : "NoStimulus",
        "stim_pars" : {},
        "drive_rate" : 3.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "SpontActivity4.5" : {
        "pattern" : "NoStimulus",
        "stim_pars" : {},
        "drive_rate" : 4.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "Sinusoidal2" : {
        "pattern" : "Sinusoidal",
        "stim_pars" : {
            "stim_start" : 1000,
            "stim_end" : 2000,
            "stim_magnitude" : 1,
            "freq" : 2,
            "offset" : 1
        },
        "drive_rate" : 1.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "Sinusoidal5" : {
        "pattern" : "Sinusoidal",
        "stim_pars" : {
            "stim_start" : 1000,
            "stim_end" : 2000,
            "stim_magnitude" : 1,
            "freq" : 5,
            "offset" : 1
        },
        "drive_rate" : 1.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "Sinusoidal10" : {
        "pattern" : "Sinusoidal",
        "stim_pars" : {
            "stim_start" : 1000,
            "stim_end" : 2000,
            "stim_magnitude" : 1,
            "freq" : 10,
            "offset" : 1
        },
        "drive_rate" : 1.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "Pulse1" : {
        "pattern" : "PulseTrain",
        "stim_pars" : {
            "stim_start" : 1000,
            "stim_duration" : 500,
            "stim_magnitude" : 1
        },
        "drive_rate" : 2.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
    "TwoSidedGaussian1" : {
        "pattern" : "TwoSidedGaussian",
        "stim_pars" : {
            "center" : 1000,
            "sigma_left" : 100,
            "sigma_right" : 200,
            "stim_magnitude" : 1,
            "offset" : 0
        },
        "drive_rate" : 2.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    }
}

In [ ]:
runner = rw.WorkflowRunner(sim_name="DiVolo-Static",
                           snn_params="divolo-static_network.yaml",
                           workflow_params=workflow_params,
                           stimuli_params="divolo-static_stimuli.yaml",
                           neuron_data_file_mark="divolo-static",
                           results_dir="divolo-static_results",
                           )

runner.add_mf_model(
    mf_name="MF-Paper",
    network_params = "divolo-static_network.yaml",
    mf_model_pars={
        "tvb_model" : "Zerlaut_adaptation_second_order",
    },
    tf_sim_pars={
        "adaptation" : True,
        "fit_transfer_function" : False,
    },
)

runner.add_mf_model(
    mf_name="MF-DiVolo",
    network_params = "divolo-static_network.yaml",
    mf_model_pars={
        "tvb_model" : "Zerlaut_adaptation_second_order",
    },
    tf_sim_pars={
        "adaptation" : True,
        "fit_transfer_function" : True,
    },
)

runner.add_mf_model(
    mf_name="MF-STP",
    network_params = "divolo-static_network.yaml",
    mf_model_pars={
        "tvb_model" : "DiVolo_STP_second_order",
    },
    tf_sim_pars={
        "adaptation" : True,
        "fit_transfer_function" : True,
    },
)

runner.add_mf_model(
    mf_name="MF-Tsodyks",
    network_params = "divolo-static_network.yaml",
    mf_model_pars={
        "tvb_model" : "DiVolo_Tsodyks_second_order",
    },
    tf_sim_pars={
        "adaptation" : True,
        "fit_transfer_function" : True,
    },
)


# Single cell simulations
First we can simulate single neurons, to get data for fitting Transfer Functions later.

The following cell will simulate neuron and create a predefined fig

In [ ]:
runner.simulate_neurons(plot=True, save=True)
Image(filename=str(runner.results_path) +'/neuron_activity.png')

To make the fig more custom one can give extra parameters to the following function

In [ ]:
fig_plots.fig_neuron_activity(
    runner.neuron_results, 
    common_params={
        'xlim' : (0,7),
        'ylim' : (0, 35)
        # 'logend' : {'loc': 'upper right'}
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': runner.results_path / f"neuron_activity2.png",
    })

Image(filename=str(runner.results_path) +'/neuron_activity2.png')

# Transfer Function

In [ ]:
runner.fit_transfer_functions(plot=True)
Image(filename=str(runner.results_path) +'/tf_fits.png') 

In [ ]:
fig_plots.fig_tf_fits_together(
    runner.neuron_results, 
    runner.tf_funcs, 
    common_params={
        'labels' : runner.mf_names,
        'linestyles' : ['--', '-.', ':', '-'],
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': runner.results_path / f"tf_fits2.png",
    })

Image(filename=str(runner.results_path) +'/tf_fits2.png') 


# Network simulations

In [ ]:
stim_name = "TestStimulus"
stimulus = {
    "pattern" : "NoStimulus",
    "stim_pars" : {},
    "drive_rate" : 3.5,
    "drive_increase_duration" : 400,
    "stim_target_ratio" : 1.0,
    "simulation_time" : 2000,
    "target_nodes" : 0,
    "direct_stimulation" : False
}

snn_results, mf_results_list = runner.simulate_single_stimulus(stimulus_name=stim_name, stimulus=stimulus, try_load=False, plot=True)

In [ ]:
Image(filename=str(runner.results_path) +f"/{stim_name}_Full_network_overview_together.png")


In [ ]:
fig_plots.fig_full_network_overview_per_column(
    snn_results, 
    mf_results_list, 
    common_params={
        'xlim': (500, 2000),
        'xmargin': 0.0,
        'ymargin': 0.0,
        'labels': runner.mf_names,
        'legend': {'loc': 'upper right', },
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (30, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': runner.results_path / f"{stim_name}_full_network_overview_per_column_{runner.data_file_mark}.png",
        'title': f"Full network overview for '{runner.data_file_mark}' with stimulus: '{stim_name}' and drive rate: {stimulus['drive_rate']} Hz",
    })

Image(filename=str(runner.results_path) +f"/{stim_name}_full_network_overview_per_column_{runner.data_file_mark}.png")


In [ ]:
reload(ax_plt)


In [ ]:
snn_results.stim_params['pattern'] == 'NoStimulus'

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20,5))

# TODO: I have no idea how to properly rescale the histograms!
#   denstity=True does not help, MF underestimates the variance and the normal distrivution is too narrow

# TODO: Derive MF params so that we can fit the voltage as well
# TODO: add MF into adaptation plot, there is only the mean (no std is computed)
# TODO: count is misleading because I do not measure all the neurons, just a subset!! 
#       so it wont sum up to the total number of neurons!!

ax_plt.FiringRateHistogramPlot({
    'binsize': 1,
    'labels' : ['SNN'] + runner.mf_names,
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[0], [snn_results]+mf_results_list)

ax_plt.VoltageHistogramPlot({
    'binsize': 0.4,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[1], [snn_results])

ax_plt.AdaptationHistogramPlot({
    'binsize': 8,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[2], [snn_results])

ax_plt.ExcitatoryNeuronConductanceHistogramPlot({
    'binsize': 0.001,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[3], [snn_results])

ax_plt.InhibitoryNeuronConductanceHistogramPlot({
    'binsize': 0.001,
    'labels' : ['SNN'],
    'start_time': 1000.0,
    'legend': True,
}).draw(axes[4], [snn_results])

fig.suptitle(f"Histograms for stimulus: '{stim_name}' and drive rate: {stimulus['drive_rate']} Hz", fontsize=16)
fig.tight_layout()
plt.show()


In [ ]:
class TestClass:
    def __init__(self, snn_results):
        self.times = snn_results.times
        self.exc_spikes_all = snn_results.exc_spikes_all
        self.inh_spikes_all = snn_results.inh_spikes_all

    def calculate_synchrony(self, population:str|list[str], start_time=0, end_time=None, spikes_threshold=5, time_bin=10):
        """Calculate the synchrony measure for the excitatory population.

        This method calculates synchrony based on pairwise correlations of spike trains.

        
        
        Parameters
        ----------
        population (str or list of str): Population(s) for which to calculate synchrony. 
            Can be 'exc', 'inh', or a list containing any combination of these.
        start_time (float): Time (in milliseconds) to start counting spikes. 
            Defaults to the constant START_TIME.
        end_time (float): Time (in milliseconds) to end counting spikes. 
            Defaults to the end of the simulation.
        spikes_threshold (int): Minimum number of spikes within the spike train to consider for statistics.

        Returns
        -------
        float or list of float: Synchrony measure(s) for the specified population(s).
        """
        if spikes_threshold < 2:
            raise ValueError("spikes_threshold must be at least 2 to calculate ISI.")

        if (end_time is None) or (end_time > self.times[-1]):
            end_time = self.times[-1]
        if start_time < self.times[0]:
            start_time = self.times[0]

        num_bins = int(round((end_time - start_time)/time_bin))
        r = start_time, end_time

        if isinstance(population, str):
            population = [population]
            unpack = True
        else:
            unpack = False

        synchrony = []
        for pop in population:
            if pop.lower() == "exc":
                population_spiketrains = self.exc_spikes_all
            elif pop.lower() == "inh":
                population_spiketrains = self.inh_spikes_all
            else:
                raise ValueError(f"Unknown population: {pop}. Valid options are 'exc' and 'inh'.")

            selected_spiketrains = []
            for spiketrain in population_spiketrains:
                spikes = []
                for spike in spiketrain:
                    if start_time <= spike <= end_time:
                        spikes.append(spike)
                selected_spiketrains.append(np.array(spikes))

            psths = [np.histogram(spikes, bins=num_bins, range=r)[0] for spikes in selected_spiketrains if len(spikes) >= spikes_threshold]
            corrs = np.nan_to_num(np.corrcoef(np.squeeze(psths)))
            synchrony.append(np.mean(corrs[np.triu_indices(len(psths), 1)]))

        if unpack:
            return synchrony[0]
        return synchrony

    def calculate_regularity(self, population:str|list[str], start_time=0, end_time=None, spikes_threshold=5):
        """Calculate the regularity measure for the excitatory population.

        This method calculates regularity based on the coefficient of variation (CV)
        of inter-spike intervals (ISIs) for each neuron in the excitatory population.

        values close to 0 -> regular firing
        values close to 1 -> Poisson firing (irregular - independent)
        values > 1 -> bursty firing

        Parameters
        ----------
        population (str or list of str): Population(s) for which to calculate regularity. 
            Can be 'exc', 'inh', or a list containing any combination of these.
        start_time (float): Time (in milliseconds) to start counting spikes. 
            Defaults to the constant START_TIME.
        end_time (float): Time (in milliseconds) to end counting spikes. 
            Defaults to the end of the simulation.
        spikes_threshold (int): Minimum number of spikes within the spike train to consider for statistics.

        Returns
        -------
        float or list of float: Regularity measure(s) for the specified population(s).
        """
        if spikes_threshold < 2:
            raise ValueError("spikes_threshold must be at least 2 to calculate ISI.")

        if (end_time is None) or (end_time > self.times[-1]):
            end_time = self.times[-1] 
        if start_time < self.times[0]:
            start_time = self.times[0]

        if isinstance(population, str):
            population = [population]
            unpack = True
        else:
            unpack = False

        regularity = []

        for pop in population:
            if pop.lower() == "exc":
                population_spiketrains = self.exc_spikes_all
            elif pop.lower() == "inh":
                population_spiketrains = self.inh_spikes_all
            else:
                raise ValueError(f"Unknown population: {pop}. Valid options are 'exc' and 'inh'.")

            selected_spiketrains = []
            for spiketrain in population_spiketrains:
                spikes = []
                for spike in spiketrain:
                    if start_time <= spike <= end_time:
                        spikes.append(spike)
                selected_spiketrains.append(np.array(spikes))
            isis = [np.diff(spikes) for spikes in selected_spiketrains if len(spikes) >= spikes_threshold]
            cvs = [np.std(isi) / np.mean(isi) for isi in isis]
            regularity.append(np.mean(cvs))                

        if unpack:
            return regularity[0]
        return regularity
    

my_tester = TestClass(snn_results)
print(my_tester.calculate_synchrony(population=['exc', 'inh'], start_time=1000.0, end_time=2000.0, spikes_threshold=5, time_bin=10))
print(my_tester.calculate_regularity(population='exc', start_time=1000.0, end_time=2000.0, spikes_threshold=5))

In [ ]:
# Regularity
from statistics import mean


start_time = 1000.0  # ms
time_bin = 10  # ms

selected_spikes = []
isi_means = []
isi_stds = []
cvs = []
psths = []

num_bins = int(round((snn_results.times[-1] - start_time)/time_bin))
r = start_time, snn_results.times[-1]


for spike_train in snn_results.exc_spikes_all:
    spikes = []
    for spike in spike_train:
        if spike >= start_time:
            spikes.append(spike)
    selected_spikes.append(spikes)
    isis = np.diff(np.array(spikes))
    if len(isis) > 1:
        isi_means.append(np.mean(isis))
        isi_stds.append(np.std(isis))
        cvs.append(np.std(isis) / np.mean(isis))
    if len(isis) > 1:
        psths.append(np.histogram(spikes, bins=num_bins, range=r)[0])


print("Regularity measures:", mean(isi_means), mean(isi_stds), mean(cvs))
plt.plot(cvs, 'o')
# values close to 0 -> regular firing
# values close to 1 -> Poisson firing (irregular - independent)
# values >1 -> bursty firing


corrs = np.nan_to_num(np.corrcoef(np.squeeze(psths)))
value = np.mean(corrs[np.triu_indices(len(psths), 1)])

print(f"Synchrony from spikes: {value}")

# Synchrony from voltage
# http://www.scholarpedia.org/article/Neuronal_synchrony_measures
# CSNG model uses spike times method above

mask = snn_results.times >= start_time

exc_voltage_var_global = snn_results.exc_voltage_mean[mask].var()
exc_voltage_var_neurons = snn_results.exc_voltage_all[mask, :].var(axis=0).mean()
synchrony_index = exc_voltage_var_global / exc_voltage_var_neurons
print("Synchrony index:", synchrony_index**0.5)


In [ ]:
plt.plot(snn_results.times, snn_results.ee_conductance_all.mean(axis=1))
plt.plot(snn_results.times, snn_results.ei_conductance_all.mean(axis=1))

plt.plot(snn_results.times, snn_results.ie_conductance_all.mean(axis=1))
plt.plot(snn_results.times, snn_results.ii_conductance_all.mean(axis=1))
# plt.xlim(0,300)

# NOTE: can I explain this? Why is inhibitory conductance low?
# Yes, the first input is excitatory drive, inhibitory conductance is coming from within-network activity only! So it starts when the inhibitory neurons start firing.


# Preferred plot is pick a neuron type and plot all conductances for it (eg ee and ie for excitatory neurons)



In [ ]:
import codes.utils.result_helpers as rh

In [ ]:
reload(rh)


In [ ]:
rh.compare_mf_snn_results(snn_results, 
                          mf_results_list, 
                          runner.mf_names, 
                          start_time=1000.0,
                          values_list=[
                                "exc_rate_mean", 
                                "inh_rate_mean", 
                                "exc_adaptation_mean", 
                                "exc_voltage_mean", 
                                "inh_voltage_mean"
                          ])

# it returns ((SNN-MF)**2).mean()
# meaning?




In [ ]:
runner.mf_names

In [ ]:
import codes.data_structures.single_neuron as sn
reload(sn)


In [ ]:
reload(sn)

In [ ]:
exc_simulation = runner.neuron_results['exc_neuron']
exc_theory = sn.AdExNeuronTheoreticalResults(
    neuron_name=exc_simulation.neuron_name,
    neuron_params=exc_simulation.neuron_params,
    exc_drive=exc_simulation.exc_drive_mean,
    inh_drive=exc_simulation.inh_drive_mean,
    out_rate=exc_simulation.out_rate_mean
)


In [ ]:
exc_theory.neuron_params

Plots to do:

- Activity with adaptation
    Heatmap, normal plot, adaptation
- Compare adaptation with prediction from nu_out
- MPF class, use it for plotting theory vs. simulations
    - use that class instead of the Results Theory class! (It will ensure all the computations are at one place!)
- Fitting TF plot

In the plot down there, we can see it doesn't make sense to include too high activities
DiVolo Fig 2 - fit of voltage only for really low activities (below inh 20Hz, exc 5 Hz)

- Would it make sense to even lower the fitting part? Like rates below 30 Hz?
- When fitting should I restrict to the data that is below saturation?

In [ ]:
nu_i_idx = 7

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']     

for j, (nu_i_idx, nu_i) in enumerate(utils.list_helpers.indexed_linear_sample(exc_simulation.nu_i[0], 5)):
    color = colors[j % len(colors)]
    plt.plot(exc_simulation.exc_drive_mean[:, nu_i_idx], exc_simulation.voltage_mean[:, nu_i_idx], "o", label=f'nu_i={nu_i} Hz', color=color)
    plt.plot(exc_theory.exc_drive_mean[:, nu_i_idx], exc_theory.voltage_mean[:, nu_i_idx], "-", color=color)
plt.title(f"Membrane potential: Theory x Simulation\n (theory=line, simulation=points)")
plt.xlabel("Excitation Drive (Hz)")
plt.ylabel("Voltage (mV)")
plt.legend()
plt.show()


plt.plot(exc_simulation.exc_drive_mean[:, nu_i_idx], exc_simulation.exc_conductance_mean[:, nu_i_idx]*1e3, "o", label='Excitation Drive (Simulation)')
plt.plot(exc_theory.exc_drive_mean[:, nu_i_idx], exc_theory.exc_conductance_mean[:, nu_i_idx], "-", label='Excitation Drive (Theory)')
plt.title(f"Excitation Drive vs Conductance for {exc_simulation.neuron_name} at nu_i={exc_simulation.inh_drive_mean[0,nu_i_idx]} Hz")
plt.legend()
plt.show()



# Inspect stimulus

Here I develop and test functionality to inspect stimulus

### Inspect spont stimulus - drive rate

In [ ]:
drive_rates = np.linspace(1.5, 7, 4)
param_to_inspect="stimulus.drive_rate"
inspection_results = runner.inspect_spont_activity(
    param_values=drive_rates,
    param_to_inspect=param_to_inspect
)

In [ ]:
[result.inspected_network_name for result in inspection_results]

In [ ]:
reload(ax_plt)

In [ ]:
fig, axes = plt.subplots(ncols=3, figsize=(16, 8))

plot = ax_plt.FiringRateInspectionPlot({
    "markers": ["o"]+[""]* (len(inspection_results)-1),
    "labels": [result.inspected_network_name for result in inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "Drive Rate (Hz)"
})
plot.draw(axes[0], inspection_results)

plot = ax_plt.VoltageInspectionPlot({
    "markers": ["o"]+[""]* (len(inspection_results)-1),
    "labels": [result.inspected_network_name for result in inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "Drive Rate (Hz)"
})
plot.draw(axes[1], inspection_results)

plot = ax_plt.AdaptationInspectionPlot({
    "markers": ["o"]+[""]* (len(inspection_results)-1),
    "labels": [result.inspected_network_name for result in inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "Drive Rate (Hz)"
})
plot.draw(axes[2], inspection_results)


fig.suptitle("Spontaneous Activity Inspection Results")
fig.tight_layout()
plt.show()

### Inspect spont stimulus - adaptation

In [ ]:
# NOTE: this comparison would be interesting together with the refitting of TFs
# aka to see how much is adaptation out of the TF

adaptation_inspection_results = runner.inspect_spont_activity(
    param_values=[0.02, 0.05, 0.1, 0.2],
    param_to_inspect="network.exc_neuron.neuron_params.b"
)

In [ ]:
fig, axes = plt.subplots(ncols=3, figsize=(16, 8))

plot = ax_plt.FiringRateInspectionPlot({
    "markers": ["o"]+[""]* (len(adaptation_inspection_results)-1),
    "labels": [result.inspected_network_name for result in adaptation_inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "b (nA)"
})
plot.draw(axes[0], adaptation_inspection_results)

plot = ax_plt.VoltageInspectionPlot({
    "markers": ["o"]+[""]* (len(adaptation_inspection_results)-1),
    "labels": [result.inspected_network_name for result in adaptation_inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "b (nA)"
})
plot.draw(axes[1], adaptation_inspection_results)

plot = ax_plt.AdaptationInspectionPlot({
    "markers": ["o"]+[""]* (len(adaptation_inspection_results)-1),
    "labels": [result.inspected_network_name for result in adaptation_inspection_results],
    "linestyles": [""] + [ ':', '--', '--', '--'],
    "legend": True,
    "xlabel": "b (nA)"
})
plot.draw(axes[2], adaptation_inspection_results)


fig.suptitle("Spontaneous Activity Inspection Results")
fig.tight_layout()
plt.show()

In [ ]:
print("Hello, World!")